# `align_bundle` — behaviour → 2P alignment

Takes ThorCam behaviour traces (box mean intensity + motion energy) on the camera clock,
interpolates them onto the **native 2P frame grid** of each recording, and writes one
aligned matrix per run. dF/F is **never resampled** — it defines the grid.

**Alignment and interpolation only.** No correlations, no bout detection, no sniff.

## What you supply
Per run: the behaviour `_boxtraces.npz`, the folder holding `dff.npy`, the **first and
last camera frame with the laser on**, and the **2P frame rate**.

## The mapping
Two anchors define the whole time base — the camera's frame rate is *not* an input to it:

```
D        = n_2p / p2_fps                  t-series duration, s
t_cam(f) = (f - on) * D / (off - on)      camera frame -> s since t-series start
t_2p(k)  = k / p2_fps
```

Any drift between the ThorCam and microscope oscillators is absorbed, because the two
anchors pin both ends. `cam_fps` is used **only** as an independent cross-check.

## ⚠ The one number to read before trusting the output
Every run prints `RATIO`. It compares the t-series duration measured two independent ways
— by the camera, and by the 2P metadata:

```
RATIO = ((off - on) / cam_fps) / (n_2p / p2_fps)      must be 1.000 +- 0.002
```

| RATIO | cause |
|---|---|
| ≈ 1.00 | fine |
| ≈ 2, 3, 4 | **`p2_fps` is wrong by that factor** — ThorImage frame averaging (`averageNum`) |
| ≈ 0.5 | anchors bracket half the acquisition, or two epochs in one movie |
| anything else | transposed digit in `on`/`off`, or the wrong `beh`↔`ca` pairing |

`p2_fps` sets the scale of the entire time axis. If it is wrong, everything stretches and
nothing else in this notebook will tell you. **RATIO is the guard.** The notebook also
cross-checks `p2_fps` against v6's own `params.json` when that file is present.

## Accuracy
- **origin offset ±1 camera frame (~±67 ms)**, common-mode — the frame straddling the
  laser edge is bright for an unknown fraction of its exposure.
- **scale ±1/(off−on) ≈ ±115 ppm** at 8 800 bright frames ≈ ±0.07 s over 10 min.

Good to ~±0.1 s absolute. Enough for envelope-vs-dF/F work; **not** enough to claim
frame-level ordering of behavioural onsets against neural events.

In [ ]:
# ========================= THE ONLY CELL YOU EDIT =========================
from pathlib import Path

OUT_DIR = Path("/grid/courses/data/imagcourse/GECI_Project_Analyzed/bundles")

# on  = FIRST camera frame with the laser ON   (== sync_detect's on_frame)
# off = LAST  camera frame with the laser ON   (== sync_detect's off_frame)
# p2_fps = 2P frame rate. THIS SETS THE TIME AXIS -- check RATIO in the output.
# cam_fps = ThorCam nominal rate. Cross-check ONLY; the anchors set the real rate.

RUNS = {
    "thormouse1_spont": dict(
        beh     = "/PATH/TO/<run>/proc/<run>_boxtraces.npz",
        ca      = "/PATH/TO/GECI_Project_Analyzed/<run>",     # folder holding dff.npy
        on      = None,
        off     = None,
        p2_fps  = None,
        cam_fps = 15.0,
    ),
    "thormouse2_spont": dict(
        beh     = "/PATH/TO/<run>/proc/<run>_boxtraces.npz",
        ca      = "/PATH/TO/GECI_Project_Analyzed/<run>",
        on      = None,
        off     = None,
        p2_fps  = None,
        cam_fps = 15.0,
    ),
    "brukermouse1_spont": dict(
        beh     = "/PATH/TO/<run>/proc/<run>_boxtraces.npz",
        ca      = "/PATH/TO/GECI_Project_Analyzed/<run>",
        on      = None,
        off     = None,
        p2_fps  = None,
        cam_fps = 15.0,
    ),
}

ENV_S     = 0.33   # conditioned-envelope window, s. Must match whisk_bouts if used later.
SAVE_MAT  = True   # also write .mat (non-fatal if scipy is missing)
REPO_DIR  = None   # folder containing whisk_bouts.py, for condition(). None = autodetect.

In [ ]:
import json, sys, traceback
import numpy as np

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("output ->", OUT_DIR)

# condition() is imported, never reimplemented: two copies of a filter drift apart.
# If it cannot be found, beh_env is simply absent -- raw ME and luminance are unaffected.
_cand = [REPO_DIR] if REPO_DIR else []
_cand += [Path.cwd(), Path.cwd().parent, Path.cwd() / "behavioral motif analysis",
          Path.cwd().parent / "behavioral motif analysis"]
condition = None
for _d in _cand:
    if _d and (Path(_d) / "whisk_bouts.py").exists():
        sys.path.insert(0, str(Path(_d)))
        from whisk_bouts import condition            # noqa: E402
        print(f"condition() imported from {Path(_d)/'whisk_bouts.py'}")
        break
if condition is None:
    print("NOTE: whisk_bouts.py not found -> 'beh_env' will be ABSENT from the bundles.\n"
          "      Raw motion energy and luminance are unaffected. Set REPO_DIR to fix.")

In [ ]:
def load_behaviour(npz_path):
    """_boxtraces.npz -> dict. Handles both vintages (older files lack trim/truncation
    keys) and never indexes boxes by position -- box sets differ between runs."""
    p = Path(npz_path)
    if not p.exists():
        raise FileNotFoundError(f"behaviour npz not found: {p}")
    d = np.load(p, allow_pickle=True)
    keys = set(d.files)
    for req in ("motion_energy", "traces", "box_names"):
        if req not in keys:
            raise KeyError(f"{p.name}: missing '{req}' (keys: {sorted(keys)})")

    names = [str(n) for n in d["box_names"]]
    me   = np.asarray(d["motion_energy"], dtype=np.float64)   # n_box x n_cam
    lum  = np.asarray(d["traces"],        dtype=np.float64)
    if me.shape != lum.shape:
        raise ValueError(f"{p.name}: motion_energy {me.shape} != traces {lum.shape}")
    if me.shape[0] != len(names):
        raise ValueError(f"{p.name}: {me.shape[0]} rows but {len(names)} box names")

    return dict(names=names, me=me, lum=lum, n_cam=me.shape[1], path=str(p),
                truncated=bool(d["truncated"]) if "truncated" in keys else False,
                trim_tail=int(d["trim_tail"]) if "trim_tail" in keys else 0,
                n_frames_source=int(d["n_frames_source"]) if "n_frames_source" in keys else -1,
                vintage="aug9+" if "trim_tail" in keys else "pre-trim")


def load_neural(folder):
    """v6 notebook output (loose .npy) or ca_extract's ca.npz. dF/F is [n_roi x n_2p]:
    v6 computes F0 with axis=1 and the FOV mean with axis=0, so rows are ROIs."""
    f = Path(folder)
    if not f.exists():
        raise FileNotFoundError(f"neural folder not found: {f}")

    out = {"path": str(f)}
    if (f / "dff.npy").exists():                                  # ---- v6 layout
        out["layout"] = "v6_npy"
        if not (f / "_COMPLETE").exists():
            print(f"  *** WARNING: no _COMPLETE in {f.name} -- that v6 run may not have "
                  f"finished, and its arrays may be left over from a previous run. ***")
        out["dff"] = np.load(f / "dff.npy")
        for key, fn in (("F_raw", "traces_raw.npy"), ("roi_npix", "roi_npix.npy"),
                        ("F0", "F0.npy")):
            if (f / fn).exists():
                out[key] = np.load(f / fn)
        if (f / "params.json").exists():
            out["params"] = json.loads((f / "params.json").read_text())
        if (f / "roi_map.tif").exists():                          # centroids: v6 saves none
            try:
                import tifffile as tiff
                rm = tiff.imread(f / "roi_map.tif")
                lab = [l for l in np.unique(rm) if l != 0]
                out["roi_xy"] = np.array([np.argwhere(rm == l).mean(0)[::-1] for l in lab],
                                         dtype=np.float64) if lab else np.zeros((0, 2))
            except Exception as e:
                print(f"  (roi_map.tif unreadable, roi_xy skipped: {e})")
    elif (f / "ca.npz").exists():                                 # ---- ca_extract layout
        out["layout"] = "ca_npz"
        z = np.load(f / "ca.npz", allow_pickle=True)
        out["dff"] = z["dff"]
        for k in ("F_raw", "roi_npix", "roi_xy", "F0", "frame_times", "fps"):
            if k in z.files:
                out[k] = z[k]
    else:
        raise FileNotFoundError(f"{f}: no dff.npy and no ca.npz")

    dff = np.asarray(out["dff"], dtype=np.float64)
    if dff.ndim != 2:
        raise ValueError(f"{f}: dff has shape {dff.shape}, expected 2-D")
    if dff.shape[0] > dff.shape[1]:
        raise ValueError(
            f"{f}: dff is {dff.shape[0]} x {dff.shape[1]} -- more ROIs than frames. "
            f"v6 writes [n_roi x n_frames]; this looks TRANSPOSED. Refusing to guess.")
    out["dff"], out["n_roi"], out["n_2p"] = dff, dff.shape[0], dff.shape[1]
    return out

In [ ]:
def align(name, cfg):
    """Interpolate behaviour onto the run's native 2P frame grid. Returns a dict of
    arrays + meta; raises on anything that would produce a quietly-wrong bundle."""
    on, off = cfg["on"], cfg["off"]
    p2_fps, cam_fps = cfg["p2_fps"], cfg["cam_fps"]
    for k in ("on", "off", "p2_fps", "cam_fps"):
        if cfg.get(k) is None:
            raise ValueError(f"{name}: '{k}' is None -- fill it in the config cell")
    on, off = int(on), int(off)

    B = load_behaviour(cfg["beh"])
    C = load_neural(cfg["ca"])
    n_cam, n_roi, n_2p = B["n_cam"], C["n_roi"], C["n_2p"]

    # ---- anchors ---------------------------------------------------------
    if not (0 <= on < off <= n_cam - 1):
        raise ValueError(f"{name}: need 0 <= on < off <= {n_cam-1}, got on={on} off={off}")
    span = off - on                                   # intervals, not frames

    # ---- the map ---------------------------------------------------------
    D        = n_2p / p2_fps                          # t-series duration, s
    cam_dt   = D / span                               # effective camera period, s
    t_cam    = (np.arange(n_cam) - on) * cam_dt       # defined for ALL camera frames
    t_2p     = np.arange(n_2p) / p2_fps

    # ---- the guard -------------------------------------------------------
    D_cam = span / cam_fps
    ratio = D_cam / D
    flag  = "OK" if abs(ratio - 1.0) <= 0.002 else "*** CHECK p2_fps / on / off ***"
    drift_ppm = (1.0 / cam_dt / cam_fps - 1.0) * 1e6
    print(f"  RATIO {ratio:.4f}  {flag}")
    print(f"    duration: camera {D_cam:8.2f} s   2P {D:8.2f} s   ({n_2p} frames @ "
          f"{p2_fps} Hz)")
    print(f"    effective cam fps {1/cam_dt:.4f} (nominal {cam_fps})  drift "
          f"{drift_ppm:+.0f} ppm")

    prm = C.get("params") or {}
    if prm.get("fps") and abs(prm["fps"] - p2_fps) / p2_fps > 0.01:
        raise ValueError(f"{name}: hardcoded p2_fps={p2_fps} but v6 params.json says "
                         f"fps={prm['fps']:.4f}. Resolve before continuing.")
    if prm.get("n_frames") and int(prm["n_frames"]) != n_2p:
        print(f"  *** WARNING: params.json n_frames={prm['n_frames']} but dff has {n_2p} "
              f"columns. ***")

    # ---- repair the two structurally-invalid ME samples -------------------
    me = B["me"].copy()
    if n_cam > 1:
        me[:, 0] = me[:, 1]          # frame 0 is 0 by construction (no previous frame)
    if on + 1 <= off:
        me[:, on] = me[:, on + 1]    # frame `on` holds the dark->bright laser step
    lum = B["lum"]

    # ---- no extrapolation, ever ------------------------------------------
    if not (t_cam[0] <= t_2p[0] and t_2p[-1] <= t_cam[-1]):
        raise ValueError(f"{name}: 2P grid [{t_2p[0]:.3f}, {t_2p[-1]:.3f}] is not inside "
                         f"the camera span [{t_cam[0]:.3f}, {t_cam[-1]:.3f}]")

    interp = lambda A: np.vstack([np.interp(t_2p, t_cam, row) for row in A])
    beh_me, beh_lum = interp(me), interp(lum)

    beh_env = None
    if condition is not None:
        # condition the FULL camera trace, then interpolate: moving_average pads by edge
        # replication, so conditioning a pre-cropped trace fabricates its endpoints.
        beh_env = interp(np.vstack([condition(row, 1.0 / cam_dt, ENV_S) for row in me]))

    # frames straddling a laser transition
    artifact = (t_2p < cam_dt) | (t_2p > D - cam_dt)

    blocks = [("roi", C["dff"], [f"roi_{i:04d}" for i in range(n_roi)]),
              ("me",  beh_me,  [f"me_{b}"  for b in B["names"]]),
              ("lum", beh_lum, [f"lum_{b}" for b in B["names"]])]
    if beh_env is not None:
        blocks.append(("env", beh_env, [f"env_{b}" for b in B["names"]]))
    M       = np.vstack([b[1] for b in blocks])
    M_names = [n for b in blocks for n in b[2]]

    meta = dict(run=name, on=on, off=off, span_intervals=span,
                p2_fps=p2_fps, cam_fps_nominal=cam_fps, cam_fps_effective=1 / cam_dt,
                tseries_duration_s=D, duration_from_camera_s=D_cam, ratio=ratio,
                drift_ppm=drift_ppm, n_cam=n_cam, n_2p=n_2p, n_roi=n_roi,
                n_box=len(B["names"]), box_names=B["names"],
                origin_uncertainty_s=cam_dt, scale_uncertainty_ppm=1e6 / span,
                beh_npz=B["path"], beh_vintage=B["vintage"], beh_truncated=B["truncated"],
                beh_trim_tail=B["trim_tail"], ca_folder=C["path"], ca_layout=C["layout"],
                ca_params=prm, env_s=ENV_S if beh_env is not None else None,
                conditioned=beh_env is not None,
                notes=["t=0 is the first camera frame with the laser on",
                       "dff is NOT resampled and NOT z-scored",
                       "me/lum/env are interpolated onto the 2P grid (linear)",
                       "me[:,0] and me[:,on] repaired before interpolation"])

    arrays = dict(t=t_2p, dff=C["dff"], beh_me=beh_me, beh_lum=beh_lum,
                  beh_names=np.array(B["names"]), artifact=artifact,
                  M=M, M_names=np.array(M_names))
    if beh_env is not None:
        arrays["beh_env"] = beh_env
    for k in ("F_raw", "roi_npix", "roi_xy"):
        if k in C:
            arrays[k] = np.asarray(C[k])

    bad = {k: v for k, v in arrays.items()
           if v.dtype.kind == "f" and not np.isfinite(v).all()}
    if bad:
        raise ValueError(f"{name}: non-finite values in {sorted(bad)}")
    return arrays, meta

In [ ]:
results, failed = {}, {}
for name, cfg in RUNS.items():
    print(f"\n=== {name} ===")
    try:
        arrays, meta = align(name, cfg)

        stem = OUT_DIR / f"{name}_aligned"
        tmp  = stem.with_suffix(".npz.tmp")                 # atomic: never a half file
        with open(tmp, "wb") as fh:
            np.savez_compressed(fh, meta=json.dumps(meta), **arrays)
        tmp.replace(stem.with_suffix(".npz"))
        stem.with_suffix(".json").write_text(json.dumps(meta, indent=2))
        if SAVE_MAT:
            try:
                from scipy.io import savemat
                savemat(stem.with_suffix(".mat"),
                        {**{k: v for k, v in arrays.items()}, "meta": json.dumps(meta)})
            except Exception as e:
                print(f"  (.mat skipped: {e})")

        results[name] = (arrays, meta)
        print(f"  wrote {stem.with_suffix('.npz').name}   M {arrays['M'].shape}  "
              f"({meta['n_roi']} ROIs + {len(arrays['beh_names'])} boxes x "
              f"{'3' if 'beh_env' in arrays else '2'} channels) x {meta['n_2p']} frames")
    except Exception as e:
        failed[name] = f"{type(e).__name__}: {e}"
        print(f"  FAILED -- {failed[name]}")
        traceback.print_exc()                    # one bad run must not cost the others

print("\n" + "=" * 78)
print(f"{'run':22s} {'RATIO':>7s} {'n_roi':>6s} {'n_2p':>7s} {'n_cam':>7s} "
      f"{'2P Hz':>7s} {'cam Hz':>8s} {'dur s':>8s}")
for n, (_, m) in results.items():
    print(f"{n:22s} {m['ratio']:7.4f} {m['n_roi']:6d} {m['n_2p']:7d} {m['n_cam']:7d} "
          f"{m['p2_fps']:7.3f} {m['cam_fps_effective']:8.4f} {m['tseries_duration_s']:8.1f}")
for n, e in failed.items():
    print(f"{n:22s} FAILED: {e}")
print("=" * 78)
print("Every RATIO must be 1.000 +- 0.002. Anything else: the bundle is wrong, not noisy.")

In [ ]:
# QC -- the eyeball gate. Top: the anchors on the laser trace. Bottom: 60 s of the
# aligned product. Look at both before anything downstream touches these bundles.
import matplotlib.pyplot as plt

for name, (A, m) in results.items():
    B = load_behaviour(RUNS[name]["beh"])
    lb = "laser_trigger" if "laser_trigger" in B["names"] else B["names"][0]
    lt = B["lum"][B["names"].index(lb)]

    fig, ax = plt.subplots(2, 1, figsize=(13, 6))
    ax[0].plot(lt, lw=.5, color="0.3")
    ax[0].axvline(m["on"], color="g", lw=1.2, label=f"on {m['on']}")
    ax[0].axvline(m["off"], color="r", lw=1.2, label=f"off {m['off']}")
    ax[0].set(title=f"{name} -- '{lb}' luminance, camera clock   RATIO {m['ratio']:.4f}",
              xlabel="camera frame", ylabel="mean intensity")
    ax[0].legend(loc="upper right", fontsize=8)

    t, sl = A["t"], slice(0, min(len(A["t"]), int(60 * m["p2_fps"])))
    src = A.get("beh_env", A["beh_me"])
    tag = "env" if "beh_env" in A else "raw ME"
    for i, b in enumerate(A["beh_names"]):
        y = src[i][sl]
        rng = np.ptp(y)
        ax[1].plot(t[sl], (y - y.mean()) / (rng if rng else 1) + i, lw=.7, label=str(b))
    d = A["dff"][:, sl]
    ax[1].plot(t[sl], d.mean(0) / (np.ptp(d.mean(0)) or 1) - 1.2, lw=.9, color="k",
               label="mean dF/F")
    ax[1].set(title=f"first 60 s on the 2P grid ({tag}, offset for display)",
              xlabel="s since t-series start", yticks=[])
    ax[1].legend(fontsize=7, ncol=4, loc="upper right")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"{name}_qc.png", dpi=140, bbox_inches="tight")
    plt.show()

## Output contract

`<OUT_DIR>/<run>_aligned.npz` — everything below shares one time axis, `t`.

| key | shape | |
|---|---|---|
| `t` | `[T]` | seconds since t-series start; `T = n_2p` |
| `dff` | `[n_roi × T]` | **untouched** — not resampled, not z-scored |
| `beh_me` | `[n_box × T]` | raw motion energy, interpolated |
| `beh_lum` | `[n_box × T]` | box mean intensity, interpolated |
| `beh_env` | `[n_box × T]` | conditioned log-ME envelope — **absent if `whisk_bouts` was not importable** |
| `beh_names` | `[n_box]` | verbatim from `box_extract`; **index by name, never position** |
| `artifact` | `[T]` bool | 2P frames straddling a laser transition |
| `M` | `[(n_roi + k·n_box) × T]` | the aligned matrix: `vstack(dff, beh_me, beh_lum[, beh_env])` |
| `M_names` | `[n_roi + k·n_box]` | `roi_0000` / `me_<box>` / `lum_<box>` / `env_<box>` |
| `F_raw`, `roi_npix`, `roi_xy` | | carried when the neural folder has them |
| `meta` | JSON string | anchors, both rates, RATIO, drift, uncertainties, provenance |

```python
z = np.load("thormouse1_spont_aligned.npz", allow_pickle=True)
t, dff, M = z["t"], z["dff"], z["M"]
names = [str(s) for s in z["beh_names"]]
whisk = z["beh_env"][names.index("whisker_pad")]     # by name
meta  = json.loads(str(z["meta"]))
```

## Two things to carry downstream

**Upsampling created no information.** The behavioural degrees of freedom are still at the
camera rate — roughly half the samples in `M`. Any per-frame statistic computed on `T`
samples will look about twice as significant as it is.

**`beh_me` is raw motion energy** and carries flicker out to the camera Nyquist (56% of
suprathreshold runs last a single frame). `beh_env` — log + zero-phase 330 ms envelope —
is the band-limited version and the right regressor for anything correlational. Both are
in the file; the choice is yours, but it is a choice.